# Olist Dataset — Data Cleaning & Insights Notebook
This notebook focuses on:
- Cleaning every table
- Handling missing values
- Correcting datatypes
- Removing duplicates
- Creating useful analytical tables
- Extracting business insights

(No visualizations included as requested.)

## 1. Import Libraries

In [3]:

import pandas as pd
import numpy as np


## 2. Load All Tables

In [4]:

customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
items = pd.read_csv("olist_order_items_dataset.csv")
geo = pd.read_csv("olist_geolocation_dataset.csv")
category = pd.read_csv("product_category_name_translation.csv")


## 3. Basic Table Information

In [5]:

tables = {
"customers":customers,
"orders":orders,
"products":products,
"sellers":sellers,
"reviews":reviews,
"payments":payments,
"items":items,
"geo":geo,
"category":category
}

for name,df in tables.items():
    print("\n====",name.upper(),"====")
    print("Shape:",df.shape)
    print(df.info())



==== CUSTOMERS ====
Shape: (99441, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB
None

==== ORDERS ====
Shape: (99441, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   9944

## 4. Remove Duplicate Records

In [6]:

customers = customers.drop_duplicates()
orders = orders.drop_duplicates()
products = products.drop_duplicates()
sellers = sellers.drop_duplicates()
reviews = reviews.drop_duplicates()
payments = payments.drop_duplicates()
items = items.drop_duplicates()
geo = geo.drop_duplicates()
category = category.drop_duplicates()


## 5. Missing Value Analysis

In [7]:

for name,df in tables.items():
    print("\nMissing values in",name)
    print(df.isnull().sum().sort_values(ascending=False).head(10))



Missing values in customers
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Missing values in orders
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
order_id                            0
order_purchase_timestamp            0
order_status                        0
customer_id                         0
order_estimated_delivery_date       0
dtype: int64

Missing values in products
product_category_name         610
product_description_lenght    610
product_name_lenght           610
product_photos_qty            610
product_weight_g                2
product_height_cm               2
product_length_cm               2
product_width_cm                2
product_id                      0
dtype: int64

Missing values in sellers
seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_

## 6. Clean Customers Table

In [8]:

customers['customer_city'] = customers['customer_city'].str.lower()
customers['customer_state'] = customers['customer_state'].str.upper()


## 7. Clean Sellers Table

In [9]:

sellers['seller_city'] = sellers['seller_city'].str.lower()
sellers['seller_state'] = sellers['seller_state'].str.upper()


## 8. Clean Products Table

In [10]:

products['product_category_name'] = products['product_category_name'].fillna("unknown")

products['product_name_lenght'] = products['product_name_lenght'].fillna(products['product_name_lenght'].median())
products['product_description_lenght'] = products['product_description_lenght'].fillna(products['product_description_lenght'].median())
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)


## 9. Clean Reviews Table

In [11]:

reviews['review_comment_title'] = reviews['review_comment_title'].fillna("no_title")
reviews['review_comment_message'] = reviews['review_comment_message'].fillna("no_message")


## 10. Convert Order Date Columns

In [12]:

date_cols=[

'order_purchase_timestamp',
'order_approved_at',
'order_delivered_carrier_date',
'order_delivered_customer_date',
'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col]=pd.to_datetime(orders[col])


## 11. Feature Engineering

In [13]:

orders['order_year']=orders['order_purchase_timestamp'].dt.year
orders['order_month']=orders['order_purchase_timestamp'].dt.month
orders['order_day']=orders['order_purchase_timestamp'].dt.day_name()
orders['order_hour']=orders['order_purchase_timestamp'].dt.hour


## 12. Delivery Time Calculation

In [14]:

orders['delivery_days']=(orders['order_delivered_customer_date']-
orders['order_purchase_timestamp']).dt.days


## 13. Merge Tables for Analysis

In [15]:

order_customer = orders.merge(customers,on="customer_id",how="left")

order_items = items.merge(products,on="product_id",how="left")

order_items = order_items.merge(sellers,on="seller_id",how="left")

full_data = order_items.merge(order_customer,on="order_id",how="left")

full_data = full_data.merge(payments,on="order_id",how="left")

full_data = full_data.merge(reviews,on="order_id",how="left")


## 14. Revenue per Order

In [16]:

revenue_per_order = items.groupby("order_id")['price'].sum().reset_index()

revenue_per_order.head()


,order_id,price
0,00010242fe8c5a6d1ba2dd792cb16214,58.90
1,00018f77f2f0320c557190d7a144bdd3,239.90
2,000229ec398224ef6ca0657da4fc703e,199.00
3,00024acbcdf0a6daa1e931b038114c75,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90


## 15. Top Selling Product Categories

In [17]:

top_categories = full_data.groupby("product_category_name")['price'].sum().sort_values(ascending=False).head(10)

print(top_categories)


product_category_name
beleza_saude              1301947.97
relogios_presentes        1254322.95
cama_mesa_banho           1107249.09
esporte_lazer             1029603.88
informatica_acessorios     950053.69
moveis_decoracao           772096.17
utilidades_domesticas      668880.94
cool_stuff                 664637.13
automotivo                 618395.50
ferramentas_jardim         519473.33
Name: price, dtype: float64


## 16. Top Seller Revenue

In [18]:

top_sellers = full_data.groupby("seller_id")['price'].sum().sort_values(ascending=False).head(10)

print(top_sellers)


seller_id
53243585a1d6dc2643021fd1853d8905    244627.55
4869f7a5dfa277a7dca6462dcf3b52b2    237867.23
4a3ca9315b744ce9f8e9374361493884    215825.77
fa1c13f2614d7b5c4749cbc52fecda94    203984.22
7c67e1448b00f6e969d365cea6b010ab    199688.11
7e93a43ef30c4f03f38b393420bc753a    182878.17
da8622b14eb17ae2831f4ac5b9dab84a    171784.57
7a67c85e85bb2ce8582c35f2203ad736    150749.79
1025f0e2d44d7041d6cf58b6550e0bfa    143675.53
955fee9216a65b617aa5c0531780ce60    137405.00
Name: price, dtype: float64


## 17. Most Used Payment Methods

In [19]:

payment_methods = payments['payment_type'].value_counts()

print(payment_methods)


payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


## 18. Average Review Score

In [20]:

avg_review = reviews['review_score'].mean()

print("Average Review Score:",avg_review)


Average Review Score: 4.08642062404257


## 19. Orders by State

In [21]:

orders_by_state = customers['customer_state'].value_counts()

print(orders_by_state.head(10))


customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64


## 20. Key Business Insights


Possible insights from the cleaned dataset:

1. Identify **highest revenue generating product categories**
2. Identify **top performing sellers**
3. Understand **customer geographic distribution**
4. Understand **payment preferences**
5. Analyze **delivery performance**
6. Measure **customer satisfaction using review scores**
7. Identify **peak ordering months and hours**


In [22]:
!pip install mysql-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 59.3 MB/s eta 0:00:00


In [23]:
import mysql.connector

In [24]:
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Anu@345",
    database="olist_ecommerce"
)

print("Connected")

InterfaceError: 2003: Can't connect to MySQL server on 'localhost:3306' (Errno 111: Connection refused)